# 5.4 Trying AMCL in ROS2

After the previous notebook, we are now quite knowledgeable about how Monte Carlo Localization works and all the ideas that underpin it. However, the implementation we worked with is, of course, kind of a simple one. More production-ready versions of the Monte Carlo Localization algorithm include a bunch of additional techniques that we sadly don't have time to explore in this course, such as *probabilistic beam-skipping*, *KD-Tree resampling*, or *degeneracy checks*.

One such implementation, very widely used in real robotics applications, is **AMCL** (which stands for *Adaptive* Monte Carlo Localization). Despite some of the aforementioned bells and whistles, at its core, AMCL is essentially the same method we are already familiar with. But, how well does it work? Well, luckily for us, a version of AMCL is included in the `navigation2` package in ROS2, so we can quite easily test it.

### Setting up

You will find the code you need for this test in the `robotics_ws` folder of this repository. This workspace includes the source code for several packages, which we will build. However, before we do that, we need to make sure we install all external dependencies. A nice thing about ROS is that it includes its own package manager, `rosdep`, to handle this sort of thing. From the root of the workspace, you can run:

```
sudo rosdep init
rosdep update
rosdep install --from-paths src
```

Let's also install `xterm`, to be able to dynamically open terminals for each process:
```
sudo apt install xterm
```

Once all the dependencies are installed, you can then build the local packages with

```
colcon build --symlink-install
```

Lastly, remember you need to set up the workspace for the active terminal session:

```
source install/setup.bash
```

### Running the demo

The included package `particle_filter_demo` has a prepared example to test AMCL. You can run it with

```
ros2 launch particle_filter_demo demo_warehouse.launch.py
```

This will open our trusty Rviz window to visualize what is going on, as well as an additional terminal emulator with the `keyboard_control` node. We are also launching `mvsim` to actually handle the simulation, but in `headless` mode (so, no GUI). The simulator might take a moment to start the first time you run the command, since it has to download some data to set up the scene.

<figure style="text-align:center">
  <img src="images/amcl.png" width="800" alt="">
  <figcaption>
  </figcaption>
</figure>

In this visualization we will see the map of the simulated environment, the point cloud generated from a laser scanner, and the pose of the robot. Actually, we will see **two** poses for the robot: in <font color="blue">blue</font>, the one estimated from pure odometry, and in <font color="green">green</font> the one estimated by AMCL (along with its uncertainty).  

The `keyboard_control` node just listens to your key presses and sends velocity commands to the robot as a response. You can use the arrow keys to change the linear and angular speeds of the robot, and the space bar to completely stop.

Play around with the simulation, and see how well AMCL manages to localize the robot. You can judge the quality of the estimation by how well the point cloud generated from the sensor reading aligns with the map of the environment.

### Changing parameters

Most ROS2 nodes allow you to tune their behavior by changing their parameters. AMCL is no exception, and has quite a few we can play with. You can find these parameters inside a file: `robotics_ws/src/particle_filter_demo/resources/params.yaml`

You are probably not going to be familiar with all of them. Some are very ROS-specific things (such as `TF` frames and topics), and some refer to the *extra refinements* (like beam-skipping) we were mentioning earlier. However, you will find some things that do sound familiar, such as minimum and maximum particle counts, or the `sigma_hit` parameter, which is the standard deviation of the gaussian used to generate the likelihood field.

You can try messing about with these parameters, and maybe even with some of the ones you don't know, and see what happens!

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

Let's stop playing for a moment, and **answer the following questions**:

- As you know, `mvsim` is the process that manages the robot simulation. How do you think the `keyboard_control` node is controlling the speed of the robot? Can you think of some way to test this theory using `ros2` terminal commands?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>The keyboard_control node controls the robot by publishing velocity commands to the /cmd_vel topic in ROS2. These messages are of type geometry_msgs/Twist and contain linear and angular velocity values. The mvsim simulator subscribes to this topic and moves the robot accordingly.You can test this by opening a new terminal and running:ros2 topic echo /cmd_vel</i></p>

- What happens to the two estimated poses over time? Can you explain the difference in their evolution?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Blue pose (odometry-only): Moves according to odometry alone. Over time, it drifts because small errors in wheel movement accumulate.Green pose (AMCL estimate): Uses both odometry and sensor readings (laser scans) to localize on the map. It corrects the drift and stays closer to the actual position.</i></p>

- The uncertainty of the robot pose estimated by AMCL is shown by a magenta ellipse (for the position) and a yellow triangle (for the orientation). Can you observe any difference in this uncertainty when running the algorithm with different parameters?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Changing parameters affects this uncertainty:Increasing max_particles or min_particles → smaller, more stable uncertainty.Increasing sigma_hit → AMCL trusts the sensor less, so uncertainty grows.Decreasing sigma_hit → AMCL trusts the sensor more, so uncertainty shrinks if measurements match the map.Changing resampling parameters affects how quickly uncertainty adapts.</i></p>